# House Prices - Exploratory Data Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)

In [ ]:
train = pd.read_csv("dataset/train.csv")
test = pd.read_csv("dataset/test.csv")

print(f"train shape: {train.shape}")
print(f"test shape: {test.shape}")
train.head()

In [ ]:
train.info()

In [ ]:
train.describe()

## Missing values

In [ ]:
missing = train.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(train) * 100).round(1)

pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})

## Target: SalePrice

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(train["SalePrice"], kde=True, ax=axes[0])
axes[0].set_title("SalePrice distribution")

sns.histplot(np.log1p(train["SalePrice"]), kde=True, ax=axes[1])
axes[1].set_title("log1p(SalePrice) distribution")

plt.tight_layout()
plt.show()

train["SalePrice"].describe()

## Correlation with SalePrice

In [ ]:
numeric_cols = train.select_dtypes(include=[np.number]).columns
corr = train[numeric_cols].corr()["SalePrice"].sort_values(ascending=False)
top_features = corr.index[1:11]

print(corr.head(11))

plt.figure(figsize=(8, 6))
sns.heatmap(train[top_features.tolist() + ["SalePrice"]].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Top 10 correlated features")
plt.show()

### Why this matters

The correlation heatmap is a fast way to prioritize which of the 80+ features are actually worth building on:

- **Feature selection / prioritization** — `OverallQual` (0.79) and `GrLivArea` (0.71) are far more predictive of `SalePrice` than most other columns. When time or model complexity is limited, these are the first features to keep.
- **Spotting redundancy (multicollinearity)** — `GarageCars` and `GarageArea` move almost identically (both describe garage size). Feeding both into a linear model inflates coefficient variance and makes them individually unstable, even though the model's overall predictions may still be fine. Tree-based models like the RandomForest below are largely unaffected by this, but it's still useful to know before engineering new features on top of them.
- **Sanity-checking the data** — a variable you *expect* to matter (e.g. total square footage) showing near-zero correlation is a signal to check for data issues, or that its relationship with price is non-linear rather than absent.
- **A cheap baseline before modeling** — comparing correlation strength against the RandomForest's `feature_importances_` later is a good gut-check: if the two disagree sharply, it often means non-linear interactions the correlation matrix can't see.

**Limitation to keep in mind:** these are Pearson correlations, so they only capture *linear* relationships. A feature with a strong curved relationship to price (e.g. diminishing returns on lot size) can show a low correlation here while still being highly predictive.

## Baseline model

Quick sanity-check model using only numeric features with median imputation — not tuned, just to confirm the pipeline works end-to-end.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

feature_cols = [c for c in numeric_cols if c not in ("Id", "SalePrice")]
X = train[feature_cols]
y = train["SalePrice"]

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

imputer = SimpleImputer(strategy="median")
X_train_imp = imputer.fit_transform(X_train)
X_valid_imp = imputer.transform(X_valid)

model = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
model.fit(X_train_imp, y_train)

preds = model.predict(X_valid_imp)
rmse = mean_squared_error(y_valid, preds) ** 0.5
print(f"Validation RMSE: {rmse:,.0f}")